# 02 — Detection Methods

Train each AnomalyShield detector on the sample data, visualize predictions,
and compare which points each method flags as anomalous.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data.loader import TimeSeriesLoader
from src.data.preprocessor import Preprocessor
from src.models.isolation_forest import IsolationForestDetector
from src.models.lof import LOFDetector
from src.models.elliptic_envelope import EllipticEnvelopeDetector
from src.models.autoencoder import AutoencoderDetector
from src.models.prophet_model import ProphetForecaster
from src.utils import set_random_seeds

set_random_seeds(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Load data
df_raw = pd.read_csv("../assets/sample_data.csv", parse_dates=["date"])
df_raw = df_raw.set_index("date").sort_index()

df = TimeSeriesLoader.from_csv("../assets/sample_data.csv")

# Ground truth: convert is_anomaly (0/1) to sklearn convention (-1/1)
y_true = np.where(df_raw["is_anomaly"] == 1, -1, 1)

print(f"Data points: {len(df)}")
print(f"Ground-truth anomalies: {(y_true == -1).sum()}")

In [ ]:
# Preprocess: normalize with StandardScaler
df_scaled, scaler = Preprocessor.normalize(df, method="standard")

# Prepare numpy array for sklearn-based detectors (n_samples, 1)
X = df_scaled[["value"]].values

print(f"Scaled data shape: {X.shape}")
print(f"Mean: {X.mean():.6f}, Std: {X.std():.6f}")

In [ ]:
# --- Isolation Forest ---
iforest = IsolationForestDetector(
    name="IsolationForest",
    n_estimators=100,
    contamination=0.05,
    random_state=42,
)
iforest.fit(X)
preds_if = iforest.predict(X)

print(f"Isolation Forest: {(preds_if == -1).sum()} anomalies detected")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df["value"], color="steelblue", linewidth=0.8, label="Value")
mask_if = preds_if == -1
ax.scatter(df.index[mask_if], df["value"].values[mask_if], color="red", s=30, zorder=5, label="Anomaly")
ax.set_title("Isolation Forest Predictions")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# --- Local Outlier Factor ---
lof = LOFDetector(
    name="LOF",
    n_neighbors=20,
    contamination=0.05,
    novelty=True,
)
lof.fit(X)
preds_lof = lof.predict(X)

print(f"LOF: {(preds_lof == -1).sum()} anomalies detected")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df["value"], color="steelblue", linewidth=0.8, label="Value")
mask_lof = preds_lof == -1
ax.scatter(df.index[mask_lof], df["value"].values[mask_lof], color="red", s=30, zorder=5, label="Anomaly")
ax.set_title("Local Outlier Factor Predictions")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# --- Elliptic Envelope ---
ee = EllipticEnvelopeDetector(
    name="EllipticEnvelope",
    contamination=0.05,
    random_state=42,
)
ee.fit(X)
preds_ee = ee.predict(X)

print(f"Elliptic Envelope: {(preds_ee == -1).sum()} anomalies detected")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df["value"], color="steelblue", linewidth=0.8, label="Value")
mask_ee = preds_ee == -1
ax.scatter(df.index[mask_ee], df["value"].values[mask_ee], color="red", s=30, zorder=5, label="Anomaly")
ax.set_title("Elliptic Envelope Predictions")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# --- LSTM Autoencoder ---
# NOTE: Using small epochs (15) to keep notebook runtime reasonable.
# The autoencoder uses sliding windows internally (default window_size=30),
# so its output length differs from the input length.
ae = AutoencoderDetector(
    name="LSTMAutoencoder",
    hidden_dim=32,
    n_layers=1,
    epochs=15,
    lr=1e-3,
    threshold_percentile=95,
    window_size=30,
    batch_size=32,
    random_state=42,
)
ae.fit(X)
preds_ae = ae.predict(X)

# The autoencoder produces one prediction per window, so there are
# (n_samples - window_size + 1) predictions. Align with the original index.
ae_index = df.index[ae.window_size - 1 :]
ae_values = df["value"].values[ae.window_size - 1 :]

print(f"Autoencoder: {(preds_ae == -1).sum()} anomalies detected (over {len(preds_ae)} windows)")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df["value"], color="steelblue", linewidth=0.8, label="Value")
mask_ae = preds_ae == -1
ax.scatter(ae_index[mask_ae], ae_values[mask_ae], color="red", s=30, zorder=5, label="Anomaly")
ax.set_title("LSTM Autoencoder Predictions")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# --- Prophet ---
# Prophet operates on DataFrames with DatetimeIndex and a 'value' column.
prophet = ProphetForecaster(
    changepoint_prior_scale=0.05,
    seasonality_mode="additive",
    interval_width=0.95,
)
result_prophet = prophet.detect_anomalies(df)

prophet_anomalies = result_prophet[result_prophet["is_anomaly"] == True]
print(f"Prophet: {len(prophet_anomalies)} anomalies detected")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(result_prophet.index, result_prophet["value"], color="steelblue", linewidth=0.8, label="Actual")
ax.plot(result_prophet.index, result_prophet["yhat"], color="darkorange", linewidth=1.0, label="Prophet Forecast")
ax.fill_between(
    result_prophet.index,
    result_prophet["yhat_lower"],
    result_prophet["yhat_upper"],
    alpha=0.15,
    color="darkorange",
    label="95% Interval",
)
ax.scatter(
    prophet_anomalies.index,
    prophet_anomalies["value"],
    color="red",
    s=30,
    zorder=5,
    label="Anomaly",
)
ax.set_title("Prophet Forecast with Anomalies")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# --- Side-by-side comparison ---
# Compare which points each method flagged.
# For the autoencoder, pad the beginning with NaN to align with the full index.
pad_len = len(df) - len(preds_ae)
preds_ae_full = np.concatenate([np.full(pad_len, np.nan), preds_ae])

# Prophet uses boolean is_anomaly; convert to -1/1
preds_prophet = np.where(result_prophet["is_anomaly"].values, -1, 1)

comparison = pd.DataFrame(
    {
        "GroundTruth": y_true,
        "IsolationForest": preds_if,
        "LOF": preds_lof,
        "EllipticEnvelope": preds_ee,
        "Autoencoder": preds_ae_full,
        "Prophet": preds_prophet,
    },
    index=df.index,
)

# Summary: count anomalies per method
summary = pd.DataFrame(
    {
        "Anomalies Detected": [
            (y_true == -1).sum(),
            (preds_if == -1).sum(),
            (preds_lof == -1).sum(),
            (preds_ee == -1).sum(),
            (preds_ae == -1).sum(),
            (preds_prophet == -1).sum(),
        ]
    },
    index=["GroundTruth", "IsolationForest", "LOF", "EllipticEnvelope", "Autoencoder", "Prophet"],
)
print("Anomaly counts by method:")
display(summary)

# Visual comparison: one subplot per method
methods = {
    "Ground Truth": y_true,
    "Isolation Forest": preds_if,
    "LOF": preds_lof,
    "Elliptic Envelope": preds_ee,
    "Prophet": preds_prophet,
}

fig, axes = plt.subplots(len(methods), 1, figsize=(14, 3 * len(methods)), sharex=True)

for ax, (name, preds) in zip(axes, methods.items()):
    ax.plot(df.index, df["value"], color="steelblue", linewidth=0.6, alpha=0.6)
    mask = preds == -1
    ax.scatter(df.index[mask], df["value"].values[mask], color="red", s=20, zorder=5)
    ax.set_ylabel("Value")
    ax.set_title(name, fontsize=10, loc="left")

axes[-1].set_xlabel("Date")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()